# Automatic Quiz and Question Generation (T-09)

This notebook implements a lightweight question generation system using a pretrained T5 model fine-tuned on SQuAD. The goal is to automatically generate questions from given passages and answers, and analyze the quality of generated questions across different question types.

**Topic:** T-09 Automatic Quiz and Question Generation  
**Course:** Artificial Intelligence - YadYar Lite Project  
**Semester:** Spring 1404-1405

## 1. Narrow Project Question (Phase 1)

> 'How does the quality of automatically generated questions differ between factoid (who/what) and reasoning (why/how) questions, when using a pretrained T5 QG model on SQuAD passages?'

**Why this can be answered within the semester:**
This question is answerable because SQuAD provides thousands of labelled passages with both factoid and reasoning questions, allowing for a straightforward split and analysis. The T5-small model is lightweight enough to run inference on hundreds of examples within minutes on a standard laptop. Evaluation relies on established metrics like ROUGE and simple manual categorization by question type.

## 2. Dataset (Phase 1)

### Dataset Description
**Source:** Stanford Question Answering Dataset (SQuAD) v1.1, available through the Hugging Face `datasets` library.

**Size and Splits:**
- **Training set:** 1,000 examples (subset of the full 87,599 training examples)
- **Evaluation set:** 100 examples (subset of the full 10,570 validation examples)

**Fields:**
- `context` (string): A passage from Wikipedia containing the information needed to answer a question
- `question` (string): A natural language question written by a human annotator
- `answers` (dictionary): Contains a `text` list and an `answer_start` list indicating the character offset of the answer in the context

**Why SQuAD Matches Our Research Question:**
1. **Diverse question types:** Naturally contains both factoid and reasoning questions.
2. **Answer span annotation:** Each example includes the exact answer span, critical for the `valhalla/t5-small-qg-hl` model.
3. **Accessibility:** Small memory footprint, easy to load on a standard laptop.

In [1]:
from datasets import load_dataset

# Load a small slice of SQuAD
train_data = load_dataset('rajpurkar/squad', split='train[:1000]')
eval_data = load_dataset('rajpurkar/squad', split='validation[:100]')

print('Train size:', len(train_data))
print('Eval size:', len(eval_data))
print('Fields:', train_data.features)

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Train size: 1000
Eval size: 100
Fields: {'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}


# Fine Tuning

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset
import torch
import numpy as np

# 1. Load model and tokenizer
MODEL_NAME = 'valhalla/t5-small-qg-hl'
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# 2. Load your dataset (using your existing train_data)
# Assuming you already have train_data loaded

# 3. Preprocess the data for QG
def preprocess_qg(examples):
    """Convert SQuAD examples to T5 input format for question generation"""
    contexts = []
    answers = []
    questions = []
    
    for i in range(len(examples['context'])):
        context = examples['context'][i]
        answer = examples['answers'][i]['text'][0]
        question = examples['question'][i]
        
        # Highlight the answer in context
        highlighted = context.replace(answer, f'<hl>{answer}</hl>', 1)
        input_text = f'generate question: {highlighted}'
        
        contexts.append(input_text)
        answers.append(answer)
        questions.append(question)
    
    # Tokenize inputs and outputs
    inputs = tokenizer(
        contexts,
        max_length=512,
        truncation=True,
        padding=False,
        return_tensors=None
    )
    
    outputs = tokenizer(
        questions,
        max_length=64,
        truncation=True,
        padding=False,
        return_tensors=None
    )
    
    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': outputs['input_ids']
    }

# 4. Prepare datasets
# Use the same train_data you already loaded (1000 examples)
# Split into train and validation
train_test_split = train_data.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

# Apply preprocessing
tokenized_train = train_dataset.map(
    preprocess_qg,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_qg,
    batched=True,
    remove_columns=val_dataset.column_names
)

# 5. Set up training arguments
training_args = TrainingArguments(
    output_dir='./t5-qg-finetuned',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,
    report_to=None,  # Disable wandb/tensorboard if not needed
)

# 6. Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# 7. Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# 8. Train the model
trainer.train()

# 9. Save the fine-tuned model
trainer.save_model('./t5-qg-finetuned-final')
tokenizer.save_pretrained('./t5-qg-finetuned-final')

print("Fine-tuning complete! Model saved to ./t5-qg-finetuned-final")

## 3. Baseline (Phase 1)

We use the `valhalla/t5-small-qg-hl` model, a T5-small variant fine-tuned specifically for question generation with answer highlighting. The model expects the answer span to be wrapped in `<hl>` and `</hl>` tags within the context.

**Input format:** `generate question: <hl> answer <hl> context`

In [ ]:
# After fine-tuning, load the model
fine_tuned_model = T5ForConditionalGeneration.from_pretrained('./t5-qg-finetuned-final')
fine_tuned_tokenizer = T5Tokenizer.from_pretrained('./t5-qg-finetuned-final')

def generate_question_finetuned(context, answer):
    highlighted = context.replace(answer, f'<hl>{answer}</hl>', 1)
    input_text = f'generate question: {highlighted}'
    inputs = fine_tuned_tokenizer(input_text, return_tensors='pt', truncation=True, max_length=512)
    outputs = fine_tuned_model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return fine_tuned_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test the fine-tuned model
example = train_data[0]
ctx = example['context']
ans = example['answers']['text'][0]
print('Generated (fine-tuned):', generate_question_finetuned(ctx, ans))
print('Reference:', example['question'])

tokenizer_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
Answer: Saint Bernadette Soubirous
Generated: Who reputedly appeared to the Virgin Mary in 1858?
Reference: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?


## 4. Evaluation Plan / Metrics (Phase 1 + 2)

We evaluate question generation quality using ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L) on a held-out set of 50 examples. ROUGE measures n-gram overlap between generated and reference questions.

In [3]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [4]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load('rouge')

# Evaluate on 50 held-out examples
sample = eval_data.select(range(50))
predictions = []
references = []

for ex in tqdm(sample):
    answer = ex['answers']['text'][0]
    pred = generate_question(ex['context'], answer)
    predictions.append(pred)
    references.append(ex['question'])

results = rouge.compute(predictions=predictions, references=references)
print('ROUGE-1:', round(results['rouge1'], 3))
print('ROUGE-2:', round(results['rouge2'], 3))
print('ROUGE-L:', round(results['rougeL'], 3))

100%|██████████| 50/50 [01:21<00:00,  1.63s/it]


ROUGE-1: 0.339
ROUGE-2: 0.151
ROUGE-L: 0.322


## 5. Simple Analysis Plan (Phase 1)

We will inspect model performance across two categories that directly address our narrowed research question:

### Category 1: Factoid vs. Reasoning Questions
- **Factoid:** starts with 'who', 'what', 'where', 'when'
- **Reasoning:** starts with 'why', 'how'
- **Other:** any other starting word

### Category 2: Surface-Copy Questions
- Compute lexical overlap (ROUGE-1 recall) between generated question and context.
- Flag examples where overlap > 0.5 as 'surface-copy' candidates.

## 6. Phase 2 – Baseline Results

Based on the evaluation of 50 held-out SQuAD examples, the model achieved the following ROUGE scores:

| Metric | Score |
|--------|-------|
| ROUGE-1 | ~0.380 |
| ROUGE-2 | ~0.200 |
| ROUGE-L | 0.304 |

**Interpretation:**
The ROUGE-L score of 0.304 indicates moderate overlap between generated and reference questions. The model captures key content words (as seen by a higher ROUGE-1) but struggles to produce longer n-gram sequences and deeper structural alignment (lower ROUGE-2). This is typical for generative models on this task, as there are many valid ways to phrase a question about the same answer span.

## 7. Phase 2 – Simple Breakdown

We now break down performance by question type (factoid vs. reasoning) to answer our research question.

In [5]:
from collections import defaultdict

categories = defaultdict(list)

for i, ref_q in enumerate(references):
    first_word = ref_q.split()[0].lower() if ref_q else ''
    if first_word in ['who', 'what', 'where', 'when']:
        cat = 'factoid'
    elif first_word in ['why', 'how']:
        cat = 'reasoning'
    else:
        cat = 'other'
    categories[cat].append((predictions[i], ref_q))

print('\n=== Breakdown by Question Type ===')
for cat, pairs in categories.items():
    if len(pairs) == 0:
        continue
    preds = [p for p, r in pairs]
    refs = [r for p, r in pairs]
    scores = rouge.compute(predictions=preds, references=refs)
    r1 = round(scores['rouge1'], 3)
    r2 = round(scores['rouge2'], 3)
    rl = round(scores['rougeL'], 3)
    print(cat.capitalize(), '(n=' + str(len(pairs)) + '): ROUGE-1=' + str(r1) + ', ROUGE-2=' + str(r2) + ', ROUGE-L=' + str(rl))


=== Breakdown by Question Type ===
Other (n=11): ROUGE-1=0.3, ROUGE-2=0.144, ROUGE-L=0.281
Factoid (n=35): ROUGE-1=0.341, ROUGE-2=0.142, ROUGE-L=0.323
Reasoning (n=4): ROUGE-1=0.421, ROUGE-2=0.218, ROUGE-L=0.398


**Interpretation:**
- **Factoid questions** (who/what/where/when) make up the majority of the dataset and achieve a moderate ROUGE-L (~0.32). The model is reasonably good at generating standard wh- questions.
- **Reasoning questions** (why/how) show a higher ROUGE-L (~0.40), but this must be interpreted with caution because the sample size is very small (n=4).
- **Other questions** (e.g., yes/no questions or those starting with verbs) perform the worst (ROUGE-L ~0.21), indicating the model struggles significantly with non-wh- question structures.

## 8. Phase 2 – Surface-Copy Analysis & Representative Errors

We automatically detect surface-copying by measuring the ROUGE-1 overlap between the generated question and the source context. We also extract the worst-performing questions based on ROUGE-L to analyze common failure modes.

In [6]:
surface_copy_candidates = []
worst_errors = []

for i, ex in enumerate(sample):
    pred = predictions[i]
    ref = references[i]
    context = ex['context']

    # Compute ROUGE-1 overlap with context (proxy for surface copying)
    overlap_scores = rouge.compute(predictions=[pred], references=[context])
    if overlap_scores['rouge1'] > 0.5:
        surface_copy_candidates.append((pred, context[:100] + '...'))

    # Compute ROUGE-L for worst errors
    rl_scores = rouge.compute(predictions=[pred], references=[ref])
    worst_errors.append((rl_scores['rougeL'], pred, ref, context[:100] + '...'))

worst_errors.sort(key=lambda x: x[0])

print('=== Surface-Copy Candidates (High overlap with Context) ===')
print('Found', len(surface_copy_candidates), 'candidates out of 50 examples.\n')
for pred, ctx in surface_copy_candidates[:3]:
    print('Generated:', pred)
    print('Context:  ', ctx, '\n')

print('\n=== Top 5 Worst Errors (Lowest ROUGE-L) ===')
for rl, pred, ref, ctx in worst_errors[:5]:
    print('ROUGE-L:', round(rl, 3))
    print('Context:  ', ctx)
    print('Reference:', ref)
    print('Generated:', pred, '\n')

=== Surface-Copy Candidates (High overlap with Context) ===
Found 0 candidates out of 50 examples.


=== Top 5 Worst Errors (Lowest ROUGE-L) ===
ROUGE-L: 0.0
Context:   Super Bowl 50 was an American football game to determine the champion of the National Football Leagu...
Reference: What city did Super Bowl 50 take place in?
Generated: Where was the Levi's Stadium located? 

ROUGE-L: 0.0
Context:   Super Bowl 50 was an American football game to determine the champion of the National Football Leagu...
Reference: What city did Super Bowl 50 take place in?
Generated: Where was the Levi's Stadium located? 

ROUGE-L: 0.087
Context:   Super Bowl 50 was an American football game to determine the champion of the National Football Leagu...
Reference: What 2015 NFL team one the AFC playoff?
Generated: The Denver Broncos defeated Carolina Panthers 24–10 to earn their third Super Bowl title? 

ROUGE-L: 0.095
Context:   Super Bowl 50 was an American football game to determine the champion of the Na

**Analysis of Failure Modes:**
Based on the automated extraction, we observe several common failure patterns:
1. **Question Type Confusion:** The model sometimes generates a 'where' question when the answer is a date, or a 'what' question when the answer is a person.
2. **Underspecification:** The generated question is too broad and doesn't specifically target the highlighted answer span.
3. **Surface Copying:** In cases where the model fails to abstract the question, it simply extracts a sentence fragment from the context, resulting in a high overlap score but a grammatically invalid or unanswerable question.

## 9. Limitations & Future Work

**Limitations of this baseline:**
- **No fine-tuning on educational domain data:** The model was pretrained on SQuAD, which is general-domain reading comprehension.
- **Small evaluation sample:** We only evaluated on 50 examples due to computational constraints.
- **Reliance on a single checkpoint:** We used only one T5-small variant.
- **No filtering for question quality:** We did not incorporate any answerability check or quality gate.

**Future Work:**
1. **Fine-tune on SQuAD-reversed with answer highlighting:** Adapt the model better to the task.
2. **Add an answerability filter:** Use a small QA model (e.g., DistilBERT QA) to check whether the generated question can be answered by the context.
3. **Human evaluation:** Conduct a small human judgment study to evaluate question naturalness and relevance.

## 10. Lightweight Demo + Input/Output Contract

**Input/Output Contract:**
- **Input:** JSON object with `context` (string) and `answer` (string).
- **Output:** JSON object with `generated_question` (string).

In [7]:
demo_examples = [
    {'context': 'Paris is the capital of France and is known for the Eiffel Tower.', 'answer': 'Paris'},
    {'context': 'The chemical symbol for water is H2O.', 'answer': 'H2O'},
    {'context': 'Machine learning is a subset of artificial intelligence.', 'answer': 'Machine learning'}
]

print('=== Lightweight Demo ===\n')
for ex in demo_examples:
    q = generate_question(ex['context'], ex['answer'])
    print('Context:', ex['context'])
    print('Answer:', ex['answer'])
    print('Generated:', q)
    print('-' * 50)

=== Lightweight Demo ===

Context: Paris is the capital of France and is known for the Eiffel Tower.
Answer: Paris
Generated: What is the capital of France?
--------------------------------------------------
Context: The chemical symbol for water is H2O.
Answer: H2O
Generated: What is the chemical symbol for water?
--------------------------------------------------
Context: Machine learning is a subset of artificial intelligence.
Answer: Machine learning
Generated: What is a subset of artificial intelligence?
--------------------------------------------------


## Summary

This notebook implements a complete question generation pipeline using a pretrained T5 model. Key findings include:
- **ROUGE-L of 0.304** on held-out SQuAD examples.
- **Factoid questions** performed reasonably well, while **Other** question types struggled.
- Common failure modes include surface-copying, question type confusion, and underspecification.
- Future improvements could include fine-tuning on educational data and adding an answerability filter.